# P08 实战演练：用 Python 复现一篇实证论文


## 本节你能学到什么
- 从数据读取、变量构造到 OLS 回归的完整流程
- 调用本项目的 `empirlab` 工具包
- 把代码结果写进论文（图表导出、系数汇报格式）
- **核心案例**：Mincer 工资方程（教育年限对工资的影响）


---
## 1. 案例背景：Mincer 工资方程
Jacob Mincer（1974）提出的经典实证模型：

$$\ln(wage_i) = \alpha + \beta_1 \cdot edu_i + \beta_2 \cdot exp_i + \beta_3 \cdot exp_i^2 + \varepsilon_i$$

- $edu$：教育年限（学历对应年数）
- $exp$：工作经验年数
- $exp^2$：经验的平方项（工资随经验先升后降）
- $\ln(wage)$：取对数消除量纲，系数可解读为「每增加1单位带来的工资增长百分比」


---
## 2. 读取 / 生成数据
实际研究中你会用 `pd.read_csv('data.csv')` 读取真实数据（如 CFPS、CGSS 等调查数据）。
本例用模拟数据，完整还原真实 Mincer 方程的估计流程。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

plt.rcParams['font.family'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ──────────────────────────────────────────
# 生成模拟数据（近似 CFPS 2018 真实参数）
# ──────────────────────────────────────────
np.random.seed(2024)
N = 1000  # 样本量

edu  = np.random.choice([6, 9, 12, 15, 16, 19],
                         N, p=[0.05, 0.20, 0.35, 0.10, 0.25, 0.05])
exp  = np.clip(np.random.normal(15, 8, N), 0, 40).astype(int)
female = np.random.binomial(1, 0.48, N)  # 性别虚拟变量

# 真实参数：教育回报率 8%，经验凸效应
log_wage = (6.0
            + 0.08 * edu
            + 0.05 * exp
            - 0.001 * exp**2
            - 0.18 * female
            + np.random.normal(0, 0.5, N))

df = pd.DataFrame({
    'log_wage': log_wage,
    'edu':      edu,
    'exp':      exp,
    'exp2':     exp**2,
    'female':   female,
})

print('数据形状:', df.shape)
df.head()


---
## 3. 描述性统计


In [ ]:
# 论文中「表1：描述性统计」的标准写法
desc = df.describe().T[['count','mean','std','min','max']]
desc.columns = ['观测数','均值','标准差','最小值','最大值']
desc = desc.round(4)
desc.index = ['对数工资','教育年限','工作经验','经验平方','女性虚拟']
print('表1 描述性统计')
print(desc.to_string())


---
## 4. OLS 回归（三个模型逐步加入控制变量）


In [ ]:
# 模型1：仅教育年限
X1 = sm.add_constant(df[['edu']])
m1 = sm.OLS(df['log_wage'], X1).fit()

# 模型2：教育 + 经验（含平方）
X2 = sm.add_constant(df[['edu','exp','exp2']])
m2 = sm.OLS(df['log_wage'], X2).fit()

# 模型3：完整模型（加入性别）
X3 = sm.add_constant(df[['edu','exp','exp2','female']])
m3 = sm.OLS(df['log_wage'], X3).fit()

print('模型1 (仅教育):')
print(f'  edu 系数 = {m1.params["edu"]:.4f}, p = {m1.pvalues["edu"]:.4f}, R2 = {m1.rsquared:.4f}')
print('\n模型2 (教育+经验):')
print(f'  edu 系数 = {m2.params["edu"]:.4f}, p = {m2.pvalues["edu"]:.4f}, R2 = {m2.rsquared:.4f}')
print('\n模型3 (完整):')
print(f'  edu 系数 = {m3.params["edu"]:.4f}, p = {m3.pvalues["edu"]:.4f}, R2 = {m3.rsquared:.4f}')


---
## 5. 论文标准回归表格
论文里不会直接粘贴 `model.summary()`，而是用标准格式表格汇报：


In [ ]:
def stars(p):
    if p < 0.01: return '***'
    elif p < 0.05: return '**'
    elif p < 0.1: return '*'
    return ''

# 构建系数表（模仿论文 Table 2）
vars_show = ['edu', 'exp', 'exp2', 'female', 'const']
var_labels = {'edu':'教育年限','exp':'工作经验','exp2':'经验平方',
              'female':'女性(=1)','const':'常数项'}

rows = []
for v in vars_show:
    vk = 'const' if v == 'const' else v
    row = {'变量': var_labels[v]}
    for i, (mn, m) in enumerate([('模型1',m1),('模型2',m2),('模型3',m3)], 1):
        if vk in m.params.index:
            coef = m.params[vk]
            se   = m.bse[vk]
            p    = m.pvalues[vk]
            row[mn] = f'{coef:.4f}{stars(p)}\n({se:.4f})'
        else:
            row[mn] = '—'
    rows.append(row)

# 底部统计
rows.append({'变量':'N',   '模型1':str(int(m1.nobs)), '模型2':str(int(m2.nobs)), '模型3':str(int(m3.nobs))})
rows.append({'变量':'R²',  '模型1':f'{m1.rsquared:.4f}', '模型2':f'{m2.rsquared:.4f}', '模型3':f'{m3.rsquared:.4f}'})
rows.append({'变量':'Adj.R²','模型1':f'{m1.rsquared_adj:.4f}', '模型2':f'{m2.rsquared_adj:.4f}', '模型3':f'{m3.rsquared_adj:.4f}'})

tbl = pd.DataFrame(rows).set_index('变量')
print('表2  Mincer 工资方程 OLS 估计结果（因变量：对数工资）')
print(tbl.to_string())
print('注：括号内为标准误；*** p<0.01，** p<0.05，* p<0.1')


---
## 6. 可视化：经验与工资的倒 U 关系


In [ ]:
coef = m3.params
exp_range = np.arange(0, 41)
# 固定 edu=12（高中）、female=0，其他取均值
pred = (coef['const']
        + coef['edu'] * 12
        + coef['exp'] * exp_range
        + coef['exp2'] * exp_range**2
        + coef['female'] * 0)

# 工资最高点（极值点）
peak_exp = -coef['exp'] / (2 * coef['exp2'])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(exp_range, np.exp(pred), color='steelblue', lw=2, label='预测工资（高中学历，男性）')
ax.axvline(peak_exp, color='red', linestyle='--', alpha=0.7,
           label=f'工资峰值年龄（经验≈{peak_exp:.0f}年）')
ax.set_xlabel('工作经验（年）')
ax.set_ylabel('预测月工资（元）')
ax.set_title('Mincer 方程预测：工资与经验的倒U关系')
ax.legend()
ax.grid(linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()
print(f'收入最高点对应工作经验约 {peak_exp:.1f} 年')


---
## 7. 调用 empirlab 包（本项目核心）
完成上面的手写版后，用本项目封装好的工具一行复现：


In [ ]:
# 注意：需要先完成项目安装（pip install -e .）
try:
    from empirlab.traditional.ols import OLS, mincer_data

    # 生成标准 Mincer 数据集
    data = mincer_data(n=1000, seed=2024)

    # 拟合模型
    model = OLS(y='log_wage', x=['edu','exp','exp2','female'], data=data)
    model.fit()
    model.summary()  # 标准表格输出

except ImportError:
    print('empirlab 包未安装，请先在项目根目录运行：')
    print('    pip install -e .')
    print()
    print('安装完成后重新运行本格即可。')


---
## 8. 把结果写进论文
### 图表导出


In [ ]:
# 导出高分辨率图片（期刊投稿常用 300 dpi，毕业论文 300-600 dpi）
# fig.savefig('fig1_mincer_wage_exp.png', dpi=300, bbox_inches='tight')
# fig.savefig('fig1_mincer_wage_exp.pdf', bbox_inches='tight')  # 矢量图（最佳）

print('取消注释后运行，图片将保存到当前目录。')


### 系数汇报格式（直接复制粘贴进论文）


In [ ]:
edu_coef = m3.params['edu']
edu_se   = m3.bse['edu']
edu_p    = m3.pvalues['edu']

print('=== 论文正文写法示例 ===')
print()
print(f'回归结果显示，教育年限的系数为 {edu_coef:.3f}（标准误 {edu_se:.3f}），')
print(f'在 1% 的显著性水平下显著（p = {edu_p:.4f}），表明教育年限每增加 1 年，')
print(f'月工资约提高 {edu_coef*100:.1f}%，支持人力资本理论的预测。')


---
## 课后任务
1. 用你自己的数据（或从 CFPS/CGSS 下载的真实数据）跑一遍 Mincer 方程，
   把结果填入上面的 `stars()` 表格，截图发到学习群讨论。
2. 在模型3基础上，新增「城乡虚拟变量（urban=1）」，观察回归系数如何变化，
   用一句话解读系数含义。
3. 尝试用对数工资之外的因变量（如月收入原值）重跑，观察残差是否仍近似正态。
   - 提示：`stats.probplot(m.resid, plot=plt.gca())`


---
## 恭喜！你已完成 P00-P08 全部教程 🎉
现在你已具备：
- Python 基础语法与数据结构
- NumPy 数组运算
- Pandas 数据清洗与分组统计
- Matplotlib 数据可视化
- 统计假设检验
- **完整的实证回归分析流程**

下一步建议：
- 回到 `notebooks/traditional/` 运行项目中的各 OLS/IV/DID/RD 示例
- 阅读 `academic/` 目录下的论文写作指南（A01-A07）
- 用真实数据复现一篇你感兴趣的实证论文
